# Can we predict whether an email is spam?

This notebook builds and evaluates binary classifiers for `Class` in `data.csv`.

**Assumption:** `Class = 1` means spam and `Class = 0` means not spam. The goal is not merely to maximize accuracy: because missing spam and blocking legitimate mail have different costs, we report precision, recall, F1, PR AUC, ROC AUC, and the confusion matrix.

The workflow deliberately keeps an untouched holdout set for the final estimate. Identical feature rows are assigned to the same split to prevent duplicate leakage.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
    accuracy_score, average_precision_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score, make_scorer,
)
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
pd.set_option("display.max_columns", 12)
sns.set_theme(style="whitegrid", context="notebook")

## 1. Load and validate the data

All predictors should be numeric, the target should contain only 0 and 1, and missing values are reported before modeling.

In [ ]:
DATA_PATH = Path("data.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Put data.csv in the same directory as this notebook.")

df = pd.read_csv(DATA_PATH)
assert "Class" in df.columns, "Expected a target column named 'Class'."
assert set(df["Class"].dropna().unique()) == {0, 1}, "Class must be binary (0/1)."
assert all(pd.api.types.is_numeric_dtype(t) for t in df.dtypes), "All columns must be numeric."

summary = pd.Series({
    "rows": len(df),
    "predictors": df.shape[1] - 1,
    "missing values": int(df.isna().sum().sum()),
    "spam rows": int(df["Class"].sum()),
    "spam rate": df["Class"].mean(),
})
display(summary.to_frame("value"))
display(df.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df["Class"].value_counts().sort_index()
axes[0].bar(["Not spam (0)", "Spam (1)"], counts.values, color=["#4C78A8", "#E45756"])
axes[0].set_title("Target balance")
axes[0].set_ylabel("Emails")
for i, value in enumerate(counts.values):
    axes[0].text(i, value + 35, f"{value:,}\n({value/len(df):.1%})", ha="center")

correlations = df.corr(numeric_only=True)["Class"].drop("Class").sort_values()
important_corr = pd.concat([correlations.head(6), correlations.tail(6)]).sort_values()
colors = ["#4C78A8" if x < 0 else "#E45756" for x in important_corr]
axes[1].barh(important_corr.index, important_corr.values, color=colors)
axes[1].set_title("Strongest univariate correlations with spam")
axes[1].set_xlabel("Pearson correlation")
plt.tight_layout()
plt.show()

## 2. Guard against duplicate leakage and create a holdout

Repeated feature vectors can make a random split look artificially good if copies occur on both sides. We hash the 57 predictors and use that hash as a group ID. `StratifiedGroupKFold` approximately preserves the class ratio while keeping each duplicate group wholly on one side.

The first fold is reserved as the final test set. Model comparison happens only within the remaining training data.

In [ ]:
X = df.drop(columns="Class")
y = df["Class"].astype(int)
groups = pd.util.hash_pandas_object(X, index=False).astype(str)

duplicate_report = pd.Series({
    "exact duplicate rows": int(df.duplicated().sum()),
    "repeated feature rows": int(X.duplicated().sum()),
    "feature groups with conflicting labels": int(df.groupby(list(X.columns), dropna=False)["Class"].nunique().gt(1).sum()),
})
display(duplicate_report.to_frame("count"))

outer_split = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(outer_split.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

split_summary = pd.DataFrame({
    "rows": [len(train_idx), len(test_idx)],
    "spam_rate": [y_train.mean(), y_test.mean()],
    "unique_feature_groups": [groups_train.nunique(), groups.iloc[test_idx].nunique()],
}, index=["training", "holdout"])
display(split_summary)
assert set(groups_train).isdisjoint(set(groups.iloc[test_idx])), "Group leakage detected."
print("No identical feature vector appears in both training and holdout data.")

## 3. Compare candidate models with group-aware cross-validation

- **Most frequent class** establishes the no-skill accuracy baseline.
- **Logistic regression** is a strong, interpretable linear baseline; scaling is learned inside each fold.
- **Random forest** and **histogram gradient boosting** capture nonlinear relationships and interactions.

We select by mean **PR AUC** (average precision), which focuses on ranking spam well and is more informative than accuracy when classes are imbalanced. All preprocessing and fitting occur independently within each fold.

In [ ]:
models = {
    "Dummy (most frequent)": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE),
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "Histogram gradient boosting": HistGradientBoostingClassifier(
        max_iter=250,
        learning_rate=0.08,
        l2_regularization=1.0,
        random_state=RANDOM_STATE,
    ),
}

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": "f1",
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
}
inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 1)

rows = []
for name, model in models.items():
    scores = cross_validate(
        model, X_train, y_train, groups=groups_train,
        cv=inner_cv, scoring=scoring, n_jobs=-1,
    )
    row = {"model": name}
    for metric in scoring:
        row[f"{metric}_mean"] = scores[f"test_{metric}"].mean()
        row[f"{metric}_sd"] = scores[f"test_{metric}"].std(ddof=1)
    rows.append(row)

cv_results = pd.DataFrame(rows).set_index("model").sort_values("pr_auc_mean", ascending=False)
display(cv_results[[f"{m}_{s}" for m in scoring for s in ("mean", "sd")]].style.format("{:.3f}"))

In [ ]:
plot_metrics = ["accuracy_mean", "f1_mean", "pr_auc_mean", "roc_auc_mean"]
ax = cv_results[plot_metrics].rename(columns=lambda c: c.replace("_mean", "").upper()).plot(
    kind="bar", figsize=(11, 5), ylim=(0, 1.05), color=["#4C78A8", "#F58518", "#54A24B", "#B279A2"]
)
ax.set_title("Group-aware 5-fold cross-validation on training data")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.legend(loc="lower right", ncol=2)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 4. Final evaluation on the untouched holdout

The best model by cross-validated PR AUC is now fit once on all training rows and evaluated once on the holdout. The classification threshold remains the standard 0.5; tuning it on the test set would bias the result.

In [ ]:
best_name = cv_results.index[0]
best_model = clone(models[best_name]).fit(X_train, y_train)
test_probability = best_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)

holdout_metrics = pd.Series({
    "accuracy": accuracy_score(y_test, test_prediction),
    "balanced accuracy": balanced_accuracy_score(y_test, test_prediction),
    "precision (spam)": precision_score(y_test, test_prediction),
    "recall (spam)": recall_score(y_test, test_prediction),
    "F1 (spam)": f1_score(y_test, test_prediction),
    "PR AUC": average_precision_score(y_test, test_probability),
    "ROC AUC": roc_auc_score(y_test, test_probability),
})

print(f"Selected model: {best_name}")
display(holdout_metrics.to_frame("holdout score").style.format("{:.3f}"))
print(classification_report(y_test, test_prediction, target_names=["not spam", "spam"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_prediction, display_labels=["Not spam", "Spam"], cmap="Blues", colorbar=False, ax=axes[0]
)
axes[0].set_title("Holdout confusion matrix")
PrecisionRecallDisplay.from_predictions(y_test, test_probability, name=best_name, ax=axes[1])
axes[1].set_title("Holdout precision–recall curve")
RocCurveDisplay.from_predictions(y_test, test_probability, name=best_name, ax=axes[2])
axes[2].plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
axes[2].set_title("Holdout ROC curve")
plt.tight_layout()
plt.show()

### Uncertainty via bootstrap

The holdout is a sample, so each metric has uncertainty. The bootstrap below resamples holdout rows 2,000 times and reports percentile 95% intervals. These intervals describe sampling variation for this dataset; they do **not** account for future changes in email content or spam tactics.

In [ ]:
def bootstrap_metric_ci(y_true, probability, metric_fn, n_boot=2000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    y_array = np.asarray(y_true)
    p_array = np.asarray(probability)
    values = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_array), len(y_array))
        if np.unique(y_array[idx]).size < 2:
            continue
        values.append(metric_fn(y_array[idx], p_array[idx]))
    return np.percentile(values, [2.5, 97.5])

interval_metrics = {
    "PR AUC": lambda yt, p: average_precision_score(yt, p),
    "ROC AUC": lambda yt, p: roc_auc_score(yt, p),
    "F1 at 0.5": lambda yt, p: f1_score(yt, p >= 0.5),
}
ci_rows = []
for name, fn in interval_metrics.items():
    point = fn(np.asarray(y_test), test_probability)
    low, high = bootstrap_metric_ci(y_test, test_probability, fn)
    ci_rows.append({"metric": name, "estimate": point, "95% CI low": low, "95% CI high": high})
ci_table = pd.DataFrame(ci_rows).set_index("metric")
display(ci_table.style.format("{:.3f}"))

## 5. What drives the predictions?

Permutation importance measures how much holdout PR AUC falls when one feature is shuffled. It is model-agnostic and describes predictive reliance, not causation. Correlated predictors can share importance, so individual values may understate their combined role.

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test,
    scoring="average_precision", n_repeats=10,
    random_state=RANDOM_STATE, n_jobs=-1,
)
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": perm.importances_mean,
    "sd": perm.importances_std,
}).sort_values("importance", ascending=False).head(15)

plt.figure(figsize=(9, 6))
plot_data = importance.sort_values("importance")
plt.barh(plot_data["feature"], plot_data["importance"], xerr=plot_data["sd"], color="#4C78A8", alpha=0.9)
plt.xlabel("Decrease in holdout PR AUC when shuffled")
plt.title(f"Top permutation importances — {best_name}")
plt.tight_layout()
plt.show()
display(importance.set_index("feature").style.format("{:.4f}"))

## 6. Conclusion and deployment considerations

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, test_prediction).ravel()
dummy_accuracy = max(y_test.mean(), 1 - y_test.mean())
pr_low, pr_high = ci_table.loc["PR AUC", ["95% CI low", "95% CI high"]]

verdict = "yes" if holdout_metrics["PR AUC"] >= 0.85 and holdout_metrics["F1 (spam)"] >= 0.80 else "partly"
display(Markdown(f"""### Verdict: **{verdict.capitalize()} — the supplied features contain substantial predictive signal.**

The selected **{best_name}** achieved **{holdout_metrics['accuracy']:.1%} accuracy**, **{holdout_metrics['F1 (spam)']:.1%} spam F1**, **{holdout_metrics['PR AUC']:.3f} PR AUC** (bootstrap 95% CI {pr_low:.3f}–{pr_high:.3f}), and **{holdout_metrics['ROC AUC']:.3f} ROC AUC** on the untouched, duplicate-group-aware holdout. By comparison, always choosing the majority class would achieve only **{dummy_accuracy:.1%} holdout accuracy**. At the default threshold, the model produced **{fp} false positives** (legitimate messages flagged) and **{fn} false negatives** (spam missed).

This supports predictive feasibility on data drawn like this sample, but it is not yet a production guarantee:

- **Threshold choice:** 0.5 is only a default. Choose a threshold using business costs on validation data—typically false positives are especially costly for spam filtering.
- **Dataset shift:** spam tactics and legitimate email patterns change. Validate on newer, truly out-of-time email before deployment and monitor precision/recall over time.
- **Generalization scope:** if many rows come from the same senders, campaigns, or time periods, split by those real-world groups when identifiers become available; feature-level duplicate grouping is only a proxy.
- **Probability calibration:** assess calibration before treating scores as literal probabilities.
- **Human-safe rollout:** quarantine or rank messages initially rather than deleting them, and capture user feedback for monitoring and retraining.
"""))